# 05 - Test-time Unicode defenses

This notebook evaluates preprocessing as a deployment-time wrapper.

**Inputs:** Clean and attacked conditions, trained model artifacts, and undefended predictions.  
**Outputs:** Defended prediction partitions and an illustrative repair table.


## Setup and defense definitions

NFC and NFKC normalize text; stripping removes default-ignorable code points; full repair also reverses the curated English confusable map.


In [1]:
import os
import unicodedata
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import regex
import yaml

# repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text(encoding='utf-8'))

PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
for directory in (PATHS['predictions'], PATHS['tables'],):
    directory.mkdir(parents=True, exist_ok=True)

# Same mapping repeated for independent execution of the notebook.
HOMOGLYPH_MAP = {
    'A': ('Α', 'А'), 'B': ('Β', 'В'), 'C': ('Ϲ', 'С'), 'E': ('Ε', 'Е'),
    'H': ('Η', 'Н'), 'I': ('Ι', 'І'), 'J': ('Ј',), 'K': ('Κ', 'К'),
    'M': ('Μ', 'М'), 'N': ('Ν',), 'O': ('Ο', 'О'), 'P': ('Ρ', 'Р'),
    'S': ('Ѕ',), 'T': ('Τ', 'Т'), 'X': ('Χ', 'Х'), 'Y': ('Υ', 'У'), 'Z': ('Ζ',),
    'a': ('а',), 'c': ('с',), 'e': ('е',), 'h': ('һ',), 'i': ('і',),
    'j': ('ј',), 'k': ('κ',), 'o': ('ο', 'о'), 'p': ('р',), 's': ('ѕ',),
    'x': ('х',), 'y': ('у',),
}

CONFUSABLE_REVERSE = {candidate: source for source, candidates in HOMOGLYPH_MAP.items() for candidate in candidates}

DEFAULT_IGNORABLE = regex.compile(r'\p{Default_Ignorable_Code_Point}+')

def remove_default_ignorables(text):
    return DEFAULT_IGNORABLE.sub('', text)

def repair_confusables_english(text):
    return ''.join(CONFUSABLE_REVERSE.get(ch, ch) for ch in text)

def apply_defense(text, defense):
    if defense == 'none':
        return text
    if defense == 'nfc':
        return unicodedata.normalize('NFC', text)
    if defense == 'nfkc':
        return unicodedata.normalize('NFKC', text)
    if defense == 'strip_ignorables':
        return remove_default_ignorables(text)
    if defense == 'nfkc_strip':
        return remove_default_ignorables(unicodedata.normalize('NFKC', text))
    if defense == 'nfkc_strip_confusable':
        normalized = remove_default_ignorables(unicodedata.normalize('NFKC', text))
        return repair_confusables_english(normalized)
    raise ValueError(f'Unknown defense: {defense}')

# Same BPE tokenizer and prediction 
import tiktoken

_BPE_ENCODING = tiktoken.get_encoding('o200k_base')

def bpe_tokenize(text):
    return [f'tok_{token_id}' for token_id in _BPE_ENCODING.encode(text)]

def predict_sklearn(model, texts, labels, chunk_size=None):
    chunk_size = chunk_size or CONFIG['classical']['prediction_chunk_size']
    predictions = []
    probabilities = []
    labels_array = np.asarray(labels, dtype=int)
    classes = np.asarray(model.classes_, dtype=int)
    class_to_column = {label: idx for idx, label in enumerate(classes)}
    for start in range(0, len(texts), chunk_size):
        stop = min(start + chunk_size, len(texts))
        batch = list(texts[start:stop])
        batch_proba = model.predict_proba(batch)
        batch_pred = classes[np.argmax(batch_proba, axis=1)]
        gt_label_columns = np.array([class_to_column[int(label)] for label in labels_array[start:stop]])
        predictions.append(batch_pred)
        probabilities.append(batch_proba[np.arange(stop - start), gt_label_columns])
    return np.concatenate(predictions), np.concatenate(probabilities)

PREDICTION_COLUMNS = [
    'dataset', 'example_id', 'model', 'model_seed', 'attack', 'strength', 'attack_seed',
    'defense', 'gt_label', 'prediction', 'gt_probability', 'correct', 'changed_count',
]

def prediction_partition_path(source, dataset, model, model_seed, defense):
    return PATHS['predictions'] / source / dataset / model / f'seed_{model_seed}' / f'{defense}.parquet'

def write_prediction_partition(frame, source, dataset, model, model_seed, defense):
    path = prediction_partition_path(source, dataset, model, model_seed, defense)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame[PREDICTION_COLUMNS].to_parquet(path, index=False)
    return path


## Build defended conditions



In [2]:
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import gc
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

data = pd.read_parquet(PATHS['processed_data'])
attacks = pd.read_parquet(PATHS['attacks'])
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Same evaluation conditions as notebook 03.
def condition_frame(dataset_name):
    clean = data[(data['dataset'] == dataset_name) & (data['split'] == 'test')][['dataset', 'example_id', 'text', 'label']].copy()
    clean = clean.assign(attack='clean', strength=0.0, attack_seed=-1, changed_count=0)
    attacked = attacks[attacks['dataset'] == dataset_name][['dataset', 'example_id', 'attacked_text', 'label', 'attack', 'strength', 'attack_seed', 'changed_count']].rename(columns={'attacked_text': 'text'})
    return pd.concat([clean, attacked], ignore_index=True)


In [3]:
condition_frame("trec")

,dataset,example_id,text,label,attack,strength,attack_seed,changed_count
0,trec,trec:test:0,How far is it from Denver to Aspen ?,5,clean,0.00,-1,0
1,trec,trec:test:1,"What county is Modesto , California in ?",4,clean,0.00,-1,0
2,trec,trec:test:2,Who was Galileo ?,3,clean,0.00,-1,0
3,trec,trec:test:3,What is an atom ?,2,clean,0.00,-1,0
4,trec,trec:test:4,When did Hawaii become a state ?,5,clean,0.00,-1,0
...,...,...,...,...,...,...,...,...
37995,trec,trec:test:499,W‍hat﻿ is‌ e-co​l﻿i ?,2,mixed,0.40,37,5
37996,trec,trec:test:499,What⁠ i﻿s ｅ-co‌li﻿ ?,2,mixed,0.40,73,5
37997,trec,trec:test:499,Wh​a﻿ｔ is⁠ e⁠－c‍o⁠l﻿i﻿ ?,2,mixed,0.75,13,10
37998,trec,trec:test:499,Ｗｈa‌t is​ e​－c​ol‌i﻿ ？,2,mixed,0.75,37,10


## Reuse unchanged predictions



In [3]:
def predict_transformer(model, tokenizer, texts, labels):
    predictions, probabilities = [], []
    batch_size = CONFIG['transformer']['eval_batch_size']
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True,
                                max_length=CONFIG['transformer']['max_length'], return_tensors='pt')
            encoded = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**encoded).logits.float().cpu()
            probability = torch.softmax(logits, dim=1).numpy()
            predictions.append(probability.argmax(axis=1))
            probabilities.append(probability[np.arange(len(batch)), labels[start:start + batch_size]])
    return np.concatenate(predictions), np.concatenate(probabilities)


## Evaluate missing defense partitions


In [ ]:
#Verboily commented because there is A LOT going on in this cell

#we start with the three classical models, each using the project's root seed
models = [(name, CONFIG['project']['root_seed']) for name in ['word_tfidf', 'char_tfidf', 'bpe_tfidf']]
#DistilBERT was trained with several seeds, so we evaluate each saved model separately
models += [('distilbert', seed) for seed in CONFIG['transformer']['seeds']]
#an example appears under several attacks, strengths and attack seeds;
#these columns together identify exactly which version of it we mean
keys = ['dataset', 'example_id', 'attack', 'strength', 'attack_seed']
#count prediction files we create and files we can reuse from an earlier run
created = reused = 0

#we run defense evaluation on each dataset
for dataset_name in ['banking77', 'trec']:
    #base contains the clean test sentences AND all their attacked versions,
    #with their gt labels and attack details, before we apply any defense
    base = condition_frame(dataset_name)
    #try each defense on the same starting texts, so defenses don't pile up
    for defense in CONFIG['defenses']['names']:
        #conditions is a new dataframe with the defended text in its text column;
        #base stays as it was, and the row order, labels and attack details stay the same
        conditions = base.assign(text=base['text'].map(lambda text: apply_defense(text, defense)))
        #.ne means "not equal": True where this defense actually changed the text
        #to_numpy gives us a boolean array in row order, used to select those rows later
        #if the input text stayed the same, we can keep the model's previous answer
        changed = base['text'].ne(conditions['text']).to_numpy()

        #now evaluate each model/seed on these defended inputs
        for model_name, model_seed in models:
            #this is where predictions for this dataset/model/seed/defense are saved
            target = prediction_partition_path('defense', dataset_name, model_name, model_seed, defense)
            #if we already have that file, skip this model/seed for this defense
            if target.exists():
                reused += 1
                continue

            #raw contains this model's saved predictions
            #for both clean and attacked inputs, with no defense applied
            raw = pd.read_parquet(prediction_partition_path('raw', dataset_name, model_name, model_seed, 'none'))
            #match each base row to its saved result using the full set of keys;
            #we can't just copy raw's columns, because its rows could be in another order
            #label comes from base; gt_label comes from the saved results
            #we keep both for a moment to check that they agree
            aligned = base[keys + ['label']].merge(
                raw[keys + ['gt_label', 'prediction', 'gt_probability']],
                #left keeps every base row, sort=False keeps the left row order,
                #and one_to_one rejects duplicate key combinations on either side
                on=keys, how='left', validate='one_to_one', sort=False,
            )
            #a missing match would leave gt_label empty; also check that every
            #saved gt label agrees with the label we have for that input
            assert aligned['gt_label'].notna().all() and aligned['label'].eq(aligned['gt_label']).all(), 'missing align'
            #start with the old predictions and gt-probabilities for ALL rows;
            #copy=True gives us separate arrays we can overwrite for changed texts
            pred = aligned['prediction'].to_numpy(copy=True)
            probability = aligned['gt_probability'].to_numpy(copy=True)

            #only load and run the model if the defense changed at least one text
            if changed.any():
                #take the defended texts for changed rows; several attack versions
                #can become the same text after repair, so evaluate each (text, label) pair once
                #we include label because gt-probability depends on which label is the gt label
                unique = conditions.loc[changed, ['text', 'label']].drop_duplicates()
                #the helpers take a list of texts and an array of gt labels;
                #labels are used to pick the gt-probability from the model's probabilities
                texts, labels = unique['text'].tolist(), unique['label'].to_numpy()
                if model_name == 'distilbert':
                    #load the trained checkpoint for this dataset and training seed,
                    #plus the tokenizer saved with it
                    model_path = PATHS['models'] / 'transformer' / dataset_name / f'seed_{model_seed}' / 'final'
                    tokenizer = AutoTokenizer.from_pretrained(model_path)
                    model = AutoModelForSequenceClassification.from_pretrained(model_path)
                    #get the predicted class and gt-probability for each unique input
                    changed_pred, changed_probability = predict_transformer(model, tokenizer, texts, labels)
                else:
                    #classical models are saved as sklearn pipelines, including
                    #their vectorizer, so loading one restores the text processing too
                    model_path = PATHS['models'] / 'classical' / dataset_name / f'{model_name}.joblib'
                    model = joblib.load(model_path)
                    changed_pred, changed_probability = predict_sklearn(model, texts, labels)

                #attach the new results to the unique (text, label) pairs
                unique = unique.assign(prediction=changed_pred, gt_probability=changed_probability)
                #now give each changed row its result, including the duplicates we skipped;
                #e.g. if three rows became the same (text, label), all three get that result
                expanded = conditions.loc[changed, ['text', 'label']].merge(
                    #many_to_one allows repeated pairs on the left but only one result
                    #per pair on the right; the left merge preserves the changed rows' order
                    unique, on=['text', 'label'], how='left', validate='many_to_one', sort=False,
                )
                #put these results back into the full arrays at the changed positions;
                #unchanged rows keep their saved predictions and gt-probabilities
                pred[changed] = expanded['prediction']
                probability[changed] = expanded['gt_probability']
                #release the loaded model before moving on to the next one
                del model
                gc.collect()
                #on a Mac with MPS which i used, also release unused memory in PyTorch's GPU cache
                if torch.backends.mps.is_available():
                    torch.mps.empty_cache()

            #build the output dataframe for ALL conditions, including unchanged texts;
            #we omit the text itself, keep the input/attack details, and use gt_label
            #as the column name to match the other saved prediction files
            frame = conditions.drop(columns='text').rename(columns={'label': 'gt_label'}).assign(
                #record which model, training seed and defense produced these results
                model=model_name, model_seed=model_seed, defense=defense,
                #these arrays now combine reused results with any newly computed ones
                prediction=pred, gt_probability=probability,
            )
            #.eq means "equal": True if the prediction matches the gt label
            frame['correct'] = frame['gt_label'].eq(frame['prediction'])
            #save this combination's results, then count one more completed file
            write_prediction_partition(frame, 'defense', dataset_name, model_name, model_seed, defense)
            created += 1

#these counts refer to saved prediction files, not individual examples
print(f'Defense partitions: {created} created, {reused} reused.')



Defense partitions: 0 created, 60 reused.


In [12]:
examples = attacks[(attacks['strength'] == 0.75) & (attacks['changed_count'] > 0)]
examples = examples.groupby('attack', sort=False).head(1)[['attack', 'clean_text', 'attacked_text']].copy()
examples['repaired_text'] = examples['attacked_text'].map(lambda text: apply_defense(text, 'nfkc_strip_confusable'))
examples


,attack,clean_text,attacked_text,repaired_text
12,homoglyph,How do I locate my card?,Нow dо I loсаtе mу саrd?,How do I locate my card?
27,invisible,How do I locate my card?,Ho﻿w d‍o⁠ I‌ l​o﻿c‌a⁠t﻿e m‌y﻿ c‍a⁠rd​?,How do I locate my card?
42,compatibility,How do I locate my card?,Ｈoｗ ｄｏ Ｉ ｌｏcａｔe mｙ ｃａｒｄ?,How do I locate my card?
72,mixed,How do I locate my card?,Ｈow​ d​o﻿ I l﻿оc‌a⁠t﻿e​ ｍy ca‌r​d？,How do I locate my card?


## Interpretation

Defense recovery must be considered alongside the change in clean performance. NFKC and ignorable stripping follow Unicode properties, whereas confusable repair encodes English-specific assumptions and can alter legitimate text in other languages or domains.